In [1]:
# ---
# title: "Getting Started with AI"
# author: "John Doe"
# date: "2024-01-15"
# tags: ["ai", "machine-learning", "tutorial"]
# difficulty: "beginner"
# ---

# # Getting Started with AI

# This is the main content of the document written in **Markdown**.

# You can include code blocks, links, and other formatting here.

In [2]:
import frontmatter

with open('example.md', 'r', encoding='utf-8') as f:
    post = frontmatter.load(f)

In [3]:
print(post.metadata['title']) # "Getting started with AI"
print(post.metadata['tags'])  # ["ai", "machine-learning", "tutorial"]

Getting Started with AI
['ai', 'machine-learning', 'tutorial']


In [4]:
print(post.content) # The markdown content without frontmatter

# Getting Started with AI

This is the main content of the document written in **Markdown**.

You can include code blocks, links, and other formatting here.


In [5]:
post.to_dict() # Access the metadata and content ath the same time

{'title': 'Getting Started with AI',
 'author': 'John Doe',
 'date': '2024-01-15',
 'tags': ['ai', 'machine-learning', 'tutorial'],
 'difficulty': 'beginner',
 'content': '# Getting Started with AI\n\nThis is the main content of the document written in **Markdown**.\n\nYou can include code blocks, links, and other formatting here.'}

In [6]:
import io
import zipfile
import requests
import frontmatter

In [7]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [8]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [9]:
print(repository_data[1])

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

In [10]:
print(repository_data[1]['filename'])

faq-main/.claude/feedback.md


In [11]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md')
            or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [12]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1216
Evidently documents: 95


In [13]:
# Split by chunks
# Eg.: 0-2000, 1000-3000, 2000-4000, etc.

def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [14]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    
    # Add the same metadata as the original document to each section
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [27]:
print(len(evidently_chunks))

[{'title': 'Create Plant', 'openapi': 'POST /plants', 'filename': 'docs-main/api-reference/endpoint/create.mdx', 'section': '## Introduction to the Document\n\nThis document serves as a guide to understanding the key concepts and topics covered within. It provides an overview of the primary themes and serves as a resource for addressing frequently asked questions.'}, {'title': 'Create Plant', 'openapi': 'POST /plants', 'filename': 'docs-main/api-reference/endpoint/create.mdx', 'section': '## Key Concepts\n\nThis section outlines the fundamental ideas necessary for a comprehensive understanding of the topic. It includes definitions, explanations, and examples relevant to the core subject matter.'}, {'title': 'Create Plant', 'openapi': 'POST /plants', 'filename': 'docs-main/api-reference/endpoint/create.mdx', 'section': '## Frequently Asked Questions (FAQs)\n\nHere, we compile a list of common questions that individuals may have regarding the topics discussed in the document. Each questi

In [16]:
# Split by paragraphs

import re
text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [17]:
# Split by sections
# Heading 1, Heading 2, etc.

import re
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """

    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with capturing
        # groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1] # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections

In [18]:
sections = split_markdown_by_level(text, level=2)

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    
    # Add the same metadata as the original document to each section
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

In [28]:
print(len(evidently_chunks))

{'title': 'Introduction', 'description': 'Example section for showcasing API endpoints', 'filename': 'docs-main/api-reference/introduction.mdx', 'section': '## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```'}


In [20]:
from openai import OpenAI

openai_client = OpenAI()

def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model = 'gpt-4o-mini',
        input = messages
    )

    return response.output_text

In [21]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

In [22]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document = text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [23]:
from tqdm.auto import tqdm

evidently_chunks = []

for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = intelligent_chunking(doc_content)

    # Add the same metadata as the original document to each section
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

  0%|          | 0/95 [00:00<?, ?it/s]

In [26]:
print(len(evidently_chunks))

788
